# 01 — Baseline evaluation
Train YOLOv8n with default settings on COCO 2017 and evaluate.  
Once you're happy with the results, the training and eval calls get extracted into `scripts/train_baseline.py` and `scripts/evaluate.py`.

## 0. Setup

In [ ]:
# Install dependencies if needed
# !pip install ultralytics pycocotools matplotlib

In [ ]:
import json
import random
from pathlib import Path

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from ultralytics import YOLO

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")

In [ ]:
# Paths — adjust COCO_ROOT if you placed the dataset elsewhere
COCO_ROOT   = Path("/datasets/coco")
DATA_YAML   = Path("../data/coco.yaml")
CONFIG_YAML = Path("../configs/baseline.yaml")
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)

VAL_IMAGES  = COCO_ROOT / "images" / "val2017"
print("Val images exist:", VAL_IMAGES.exists())

## 1. Train baseline
This cell trains YOLOv8n using `configs/baseline.yaml`.  
Skip and load an existing checkpoint if you've already trained.

In [ ]:
model = YOLO("yolov8n.pt")  # downloads pretrained weights on first run

results = model.train(cfg=str(CONFIG_YAML))

# Ultralytics saves the best checkpoint automatically
BEST_PT = Path(results.save_dir) / "weights" / "best.pt"
print("Best checkpoint:", BEST_PT)

In [ ]:
# --- OR: load an existing checkpoint instead of retraining ---
# BEST_PT = Path("../runs/baseline/weights/best.pt")
# model   = YOLO(str(BEST_PT))

## 2. COCO-style evaluation

In [ ]:
model = YOLO(str(BEST_PT))

metrics = model.val(
    data=str(DATA_YAML),
    imgsz=640,
    save_json=True,   # writes predictions_coco_format.json inside run dir
    verbose=True,
)

baseline_metrics = {
    "mAP50":     round(float(metrics.box.map50), 4),
    "mAP50_95":  round(float(metrics.box.map),   4),
    "precision": round(float(metrics.box.mp),    4),
    "recall":    round(float(metrics.box.mr),    4),
}

print(json.dumps(baseline_metrics, indent=2))

# Save for the comparison notebook
out = RESULTS_DIR / "baseline_coco_eval.json"
with open(out, "w") as f:
    json.dump(baseline_metrics, f, indent=2)
print(f"Saved → {out}")

## 3. Training curves
Ultralytics writes a `results.csv` inside the run directory with per-epoch metrics.

In [ ]:
import pandas as pd

run_dir    = BEST_PT.parent.parent          # e.g. runs/baseline/
csv_path   = run_dir / "results.csv"

df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()         # Ultralytics adds whitespace

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(df["epoch"], df["train/box_loss"], label="train box loss")
axes[0].plot(df["epoch"], df["val/box_loss"],   label="val box loss")
axes[0].set_title("Box loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(df["epoch"], df["train/cls_loss"], label="train cls loss")
axes[1].plot(df["epoch"], df["val/cls_loss"],   label="val cls loss")
axes[1].set_title("Class loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()

axes[2].plot(df["epoch"], df["metrics/mAP50(B)"],    label="mAP@0.5")
axes[2].plot(df["epoch"], df["metrics/mAP50-95(B)"], label="mAP@0.5:0.95")
axes[2].set_title("Validation mAP")
axes[2].set_xlabel("Epoch")
axes[2].legend()

plt.suptitle("Baseline — YOLOv8n training curves", y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "baseline_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Per-class mAP
Which classes does the baseline struggle with most?

In [ ]:
# metrics.box.maps is a numpy array of per-class mAP50-95
class_names = model.names   # dict {int: str}
per_class   = {class_names[i]: round(float(v), 4) for i, v in enumerate(metrics.box.maps)}

# Sort by mAP ascending so worst classes appear at top of the chart
sorted_classes = sorted(per_class.items(), key=lambda x: x[1])
names, scores  = zip(*sorted_classes)

fig, ax = plt.subplots(figsize=(8, 14))
bars = ax.barh(names, scores, color=["#d9534f" if s < 0.3 else "#5cb85c" for s in scores])
ax.axvline(baseline_metrics["mAP50_95"], linestyle="--", color="grey", label=f'mean mAP = {baseline_metrics["mAP50_95"]}')
ax.set_xlabel("mAP@0.5:0.95")
ax.set_title("Baseline per-class mAP — worst to best")
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "baseline_per_class_map.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n5 worst classes:")
for name, score in sorted_classes[:5]:
    print(f"  {name:<20} {score:.4f}")

## 5. Visualize predictions on val images
Spot-check what the baseline model actually sees.

In [ ]:
CONF_THRESH = 0.25
N_IMAGES    = 6

val_imgs = sorted(VAL_IMAGES.glob("*.jpg"))
sample   = random.sample(val_imgs, N_IMAGES)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, img_path in zip(axes, sample):
    result = model.predict(str(img_path), conf=CONF_THRESH, verbose=False)[0]
    img    = Image.open(img_path).convert("RGB")
    ax.imshow(img)
    ax.axis("off")

    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        conf  = float(box.conf[0])
        cls   = int(box.cls[0])
        label = f"{model.names[cls]} {conf:.2f}"
        rect  = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=1.5, edgecolor="#00bfff", facecolor="none"
        )
        ax.add_patch(rect)
        ax.text(x1, y1 - 4, label, fontsize=7, color="#00bfff",
                bbox=dict(facecolor="black", alpha=0.4, pad=1, edgecolor="none"))

    ax.set_title(img_path.name, fontsize=8)

plt.suptitle(f"Baseline predictions (conf ≥ {CONF_THRESH})", y=1.01)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "baseline_sample_predictions.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Confidence score distribution
Useful for checking whether the model is over- or under-confident — informs whether you need label smoothing or confidence calibration.

In [ ]:
N_SAMPLE  = 200   # number of val images to run inference on
all_confs = []

sample_paths = random.sample(val_imgs, N_SAMPLE)
preds = model.predict(sample_paths, conf=0.01, verbose=False)  # low threshold to capture all boxes

for r in preds:
    all_confs.extend(r.boxes.conf.tolist())

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(all_confs, bins=50, color="steelblue", edgecolor="white")
ax.axvline(0.25, color="red",    linestyle="--", label="default conf=0.25")
ax.axvline(0.5,  color="orange", linestyle="--", label="conf=0.5")
ax.set_xlabel("Confidence score")
ax.set_ylabel("Count")
ax.set_title(f"Baseline confidence distribution ({N_SAMPLE} val images)")
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "baseline_conf_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Total boxes sampled : {len(all_confs)}")
print(f"Median confidence   : {np.median(all_confs):.3f}")
print(f"Boxes above 0.5     : {sum(c > 0.5 for c in all_confs)} ({100*sum(c>0.5 for c in all_confs)/len(all_confs):.1f}%)")

## 7. Summary
Print all numbers in one place — copy these into your README as the baseline row.

In [ ]:
print("=" * 40)
print("BASELINE — YOLOv8n on COCO val2017")
print("=" * 40)
for k, v in baseline_metrics.items():
    print(f"  {k:<15} {v:.4f}")
print()
print("Checkpoint :", BEST_PT)
print("Results dir:", RESULTS_DIR)
print()
print("Next: open 02_ablation_analysis.ipynb to compare with modified runs.")